In [1]:
import os
import sys

# 设置缓存目录路径
cache_dir = r'F:\Anaconda\envs\pytorch\cache'

# 确保目录存在
os.makedirs(cache_dir, exist_ok=True)

# 设置所有相关环境变量
os.environ['HF_HOME'] = cache_dir
os.environ['TRANSFORMERS_CACHE'] = cache_dir
os.environ['HUGGINGFACE_HUB_CACHE'] = cache_dir

print(f"设置的缓存目录: {cache_dir}")

# 现在导入transformers
from transformers import pipeline
from transformers.utils import TRANSFORMERS_CACHE

print(f"实际缓存路径: {TRANSFORMERS_CACHE}")

# 验证路径是否正确
if cache_dir == TRANSFORMERS_CACHE:
    print("✅ 缓存路径设置成功！")
else:
    print(f"❌ 缓存路径设置失败！期望: {cache_dir}, 实际: {TRANSFORMERS_CACHE}")

# 运行情感分析
classifier = pipeline("sentiment-analysis")
result = classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)

print(f"\n分析结果: {result}")

# 检查缓存目录是否创建了文件
print(f"\n缓存目录内容:")
if os.path.exists(cache_dir):
    items = os.listdir(cache_dir)
    if items:
        for item in items:
            item_path = os.path.join(cache_dir, item)
            if os.path.isdir(item_path):
                print(f"  📁 {item}")
            else:
                print(f"  📄 {item}")
    else:
        print("   (空目录)")
else:
    print("   (目录不存在)")

import os

# 检查并修复缓存目录权限
cache_dir = r"F:\Anaconda\envs\pytorch\cache"
if os.path.exists(cache_dir):
    import subprocess
    # 给当前用户完全控制权限
    subprocess.run(['icacls', cache_dir, '/grant', 'Everyone:F', '/T'], shell=True)

设置的缓存目录: F:\Anaconda\envs\pytorch\cache


f:\Anaconda\envs\pytorch\Lib\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


实际缓存路径: F:\Anaconda\envs\pytorch\cache
✅ 缓存路径设置成功！


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /distilbert/distilbert-base-uncased-finetuned-sst-2-english/resolve/714eb0f/config.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='huggingface.co', port=443) at 0x214d9452900>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 29edf599-4c30-4672-85f8-338ca857a3e1)')' thrown while requesting HEAD https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english/resolve/714eb0f/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /distilbert/distilbert-base-uncased-finetuned-sst-2-english/resolve/714eb0f/config.json (Caused by ConnectTimeoutError(<HTTPSConnection(host='huggingface.co', port=443) at 0x214d945a210>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: c27577b1-451d-428b-bbd1-6ed3d463def2)')' thrown 

Device set to use cuda:0



分析结果: [{'label': 'POSITIVE', 'score': 0.9598046541213989}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

缓存目录内容:
  📁 .locks
  📁 datasets
  📁 datasets--bigcode--the-stack
  📁 datasets--code_search_net
  📁 datasets--conll2003
  📁 datasets--glue
  📁 datasets--imdb
  📁 datasets--lewtun--github-issues
  📁 datasets--wikitext
  📁 evaluate
  📁 metrics
  📁 models--bert-base-cased
  📁 models--bert-base-uncased
  📁 models--camembert-base
  📁 models--dbmdz--bert-large-cased-finetuned-conll03-english
  📁 models--distilbert--distilbert-base-cased-distilled-squad
  📁 models--distilbert--distilbert-base-uncased-finetuned-sst-2-english
  📁 models--distilbert-base-cased-distilled-squad
  📁 models--distilbert-base-uncased
  📁 models--gpt2
  📁 models--roberta-base
  📁 models--sentence-transformers--multi-qa-mpnet-base-dot-v1
  📁 models--t5-small
  📁 models--WenjieSHI--dummy-model
  📁 models--xlnet-base-cased
  📁 modules
  📄 stored_tokens
  📄 token
  📁 xet


提取文本摘要

在本节中，我们将看看如何使用 Transformer 模型将长篇文档压缩为摘要，这项任务称为文本摘要。这是最具挑战性的自然语言处理（NLP）任务之一，因为它需要一系列能力，例如理解长篇文章并且生成能够捕捉文档中主要主题的连贯文本。但是，如果做得好，文本摘要是一种强大的工具，可以减轻各个领域的人详细阅读长文档的负担，从而加快业务流程。

尽管在 Hugging Face Hub 上已经存在各种提取文本摘要的微调模型，但是几乎所有的这些模型都只适用于英文文档。因此，为了在本节中添加一些不一样的特点，我们将为英语和西班牙语训练一个双语模型。在本节结束时，你将有一个可以总结客户评论的 模型 。

如果你试一试的话，就发现模型能够生成非常简洁的摘要，因为它们是从客户在产品评论中提供的标题中学到的。让我们首先为这项任务准备一个合适的双语语料库。

准备多语言语料库

我们将使用 多语言亚马逊评论语料库 创建我们的双语摘要器。该语料库由六种语言的亚马逊产品评论组成，通常用于多语言分类器的基准测试。然而，由于每条评论都附有一个简短的标题，我们可以使用标题作为我们模型学习的参考摘要！首先，让我们从 Hugging Face Hub 下载英语和西班牙语子集：

In [2]:
from datasets import load_dataset

spanish_dataset = load_dataset("amazon_reviews_multi", "es")
english_dataset = load_dataset("amazon_reviews_multi", "en")
english_dataset

'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /datasets/amazon_reviews_multi/resolve/main/README.md (Caused by ConnectTimeoutError(<HTTPSConnection(host='huggingface.co', port=443) at 0x21491a97890>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: f80b1d32-b5d1-4505-afb6-edbbb2f86071)')' thrown while requesting HEAD https://huggingface.co/datasets/amazon_reviews_multi/resolve/main/README.md
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /datasets/amazon_reviews_multi/resolve/main/README.md (Caused by ConnectTimeoutError(<HTTPSConnection(host='huggingface.co', port=443) at 0x21491a97c50>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: c3e95863-171a-4bb5-a98e-93b75909943f)')' thrown while requesting HEAD https://huggingface.co/datasets/amazon_reviews_multi/resolve/main/README.md
Retr

ConnectionError: Couldn't reach 'amazon_reviews_multi' on the Hub (LocalEntryNotFoundError)

如你所见，在英语数据集的 train 部分有 200,000 条评论， validation 和 test 部分有 5,000 条评论。我们感兴趣的评论正文和标题保存在 review_body 和 review_title 列中。让我们通过创建一个简单的函数来从训练集中随机抽取一些样本，该函数使用我们在 第五章 学到过：

In [ ]:
def show_samples(dataset, num_samples=3, seed=42):
    sample = dataset["train"].shuffle(seed=seed).select(range(num_samples))
    for example in sample:
        print(f"\n'>> Title: {example['review_title']}'")
        print(f"'>> Review: {example['review_body']}'")


show_samples(english_dataset)

✏️ 试试看！ 更改 Dataset.shuffle() 命令中的随机种子以探索语料库中的其他评论。如果你是说西班牙语的人，请查看 spanish_dataset 中的一些评论，看看标题是否像是合理的摘要。

这个示例显示了人们通常在网上评论的多样性，从积极的到消极的（以及介于两者之间的评论！）。尽管带有“meh”标题的示例的信息量不大，但其他标题看起来像是对评论本身的不错的总结。在单个 GPU 上训练所有 400,000 条评论的摘要模型将花费太长时间，因此我们将专注于为单个产品领域生成摘要。为了了解我们可以选择哪些领域，让我们将 english_dataset 转换为 pandas.DataFrame ，并计算每个产品类别的评论数量：

In [ ]:
english_dataset.set_format("pandas")
english_df = english_dataset["train"][:]
# 显示前 20 个产品的数量
english_df["product_category"].value_counts()[:20]

在英语数据集中，最受欢迎的产品是家居用品、服装和无线电子产品。不过，为了带有亚马逊的特色，让我们专注于总结书籍的评论——毕竟，这是亚马逊这家公司成立的基础！我们可以看到两个符合要求的产品类别（ book 和 digital_ebook_purchase ），所以让我们用这两个产品类别过滤两种语言的数据集。正如我们在 第五章 学到的， Dataset.filter() 函数可以让我们非常有效地对数据集进行切片，所以我们可以定义一个简单的函数来进行此操作：

In [ ]:
def filter_books(example):
    return (
        example["product_category"] == "book"
        or example["product_category"] == "digital_ebook_purchase"
    )

当我们使用这个函数对 english_dataset 和 spanish_dataset 过滤后，结果将只包含涉及书籍类别的那些行。在使用过滤器之前，让我们将 english_dataset 的格式从 "pandas" 切换回 "arrow" ：

In [ ]:
english_dataset.reset_format()

然后我们可以使用过滤器功能，作为一个基本的检查，让我们检查一些评论的样本，看看它们是否确实与书籍有关：

In [ ]:
spanish_books = spanish_dataset.filter(filter_books)
english_books = english_dataset.filter(filter_books)
show_samples(english_books)

好吧，我们可以看到评论并不是严格意义上的书籍，也可能是指日历和 OneNote 等电子应用程序等内容。尽管如此，该领域似乎也适合训练摘要模型。在我们查筛选适合此任务的各种模型之前，我们还有最后一点数据准备要做：将英文和西班牙文评论作为单个 DatasetDict 对象组合起来。🤗 Datasets 提供了一个方便的 concatenate_datasets() 函数，它（名如其实）将把两个 Dataset 对象堆叠在一起。因此，为了创建我们的双语数据集，我们将遍历数据集的每个部分，并打乱结果以确保我们的模型不会过度拟合单一语言：

In [ ]:
from datasets import concatenate_datasets, DatasetDict

books_dataset = DatasetDict()

for split in english_books.keys():
    books_dataset[split] = concatenate_datasets(
        [english_books[split], spanish_books[split]]
    )
    books_dataset[split] = books_dataset[split].shuffle(seed=42)

# 挑选一些样例
show_samples(books_dataset)

这的确看起来像是混合了英语和西班牙语的评论！现在我们有了一个训练语料库，最后要检查的一件事是评论及其标题中单词的分布。这对于摘要任务尤其重要，其中数据中如果出现大量参考摘要过于简短会使模型偏向于生成的摘要中仅有一两个单词。下面的图中显示了单词分布，我们可以看到有些标题严重偏向于 1-2 个单词：

为了解决这个问题，我们将过滤掉标题非常短的示例，以便我们的模型可以生成更有效的摘要。由于我们正在处理英文和西班牙文文本，因此我们可以使用粗略的启发式方法在空白处拆分标题的单词，然后用我们强大的 Dataset.filter() 方法如下：

In [ ]:
books_dataset = books_dataset.filter(lambda x: len(x["review_title"].split()) > 2)

现在我们已经准备好了我们的语料库，让我们来看看一些可以对其进行微调的可选的 Transformer 模型！

文本摘要模型

如果你仔细想想，文本摘要是一种类似于机器翻译的任务：我们有一个像评论这样的文本正文，我们希望将其“翻译”成一个较短的版本，同时捕捉到输入文本的主要特征。因此，大多数用于文本摘要的 Transformer 模型采用了我们在 第一章 遇到的编码器-解码器架构。尽管有一些例外，例如 GPT 系列模型，它们在 few-shot（少量微调）之后也可以提取摘要。下表列出了一些可以进行摘要微调的流行预训练模型。

从此表中可以看出，大多数用于摘要的 Transformer 模型（以及大多数 NLP 任务）都是单一语言的。如果你的任务所使用的语言是“有大量语料库”（如英语或德语）的语言，这很好。但对于世界各地正在使用的数千种其他语言，则不然。幸运的是，有一类多语言 Transformer 模型，如 mT5 和 mBART，可以解决问题。这些模型也是使用因果语言建模进行预训练的，但有一点不同：它们不是在一种语言的语料库上训练，而是同时在 50 多种语言的文本上进行联合训练！

我们将使用 mT5，这是一种基于 T5 的有趣架构，在文本到文本任务中进行了预训练。在 T5 中，每个 NLP 任务都是以任务前缀（如 summarize: ）的形式定义的，模型根据不同的任务生成不同的文本。如下图所示，这让 T5 变得非常通用，因为你可以用一个模型解决很多任务！

mT5 不使用前缀，但具有 T5 的大部分功能，并且具有多语言的优势。现在我们已经选择了一个模型，接下来让我们来看看如何准备我们的训练数据。

✏️ 试试看！ 完成本节后，可以尝试比较一下 mT5 和用相同技术微调过的 mBART 的性能。附加的挑战：只在英文评论上微调 T5。因为 T5 有一个特殊的前缀提示，你需要在下面的预处理步骤中将 summarize: 添加到输入例子前。

我们接下来的任务是对我们的评论及其标题进行 tokenize 和 encode 。通常，我们需要首先加载与预训练模型 checkpoint 相关的 tokenizer，这次我们将使用较小的 mt5-small 作为我们的 checkpoint 这样我们就可以在合理的时间消耗内对模型进行微调：

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

💡在 NLP 项目的早期阶段，一个好的做法是在小样本数据上训练一类“小”模型。这使你可以更快地调试和迭代端到端工作流。当你对结果有信心之后，你只需要通过简单地更改模型 checkpoint 就可以在较大规模数据上训练模型！

让我们在一个小样本上测试 mT5 tokenizer

In [ ]:
inputs = tokenizer("I loved reading the Hunger Games!")
inputs

在这里我们可以看到熟悉的 input_ids 和 attention_mask ，我们在 第3章 的第一次微调实验中遇到过。让我们使用 tokenizer 的 convert_ids_to_tokens() 函数解码这些输入 ID，看看我们正在处理的是什么类型的 tokenizer：

In [ ]:
tokenizer.convert_ids_to_tokens(inputs.input_ids)

从特殊的 Unicode 字符 ▁ 和表示序列结束 </s> token 可以看出来，我们正在使用基于 第6章 中讨论的 Unigram 子词分词算法的 SentencePiece tokenizer 。 Unigram 对于多语言语料库特别有用，因为它让 SentencePiece 不必受口音、标点符号以及很多语言（如日语）没有空白字符的影响，只专注于找出最优的分词方式。

为了对我们的语料库 tokenize ，我们需要处理与摘要任务会遇到的一个细微问题：因为我们的输出目标也是文本，所以输入和输出加起来可能超过模型的最大上下文大小。这意味着我们需要对评论及其标题进行截断，以确保我们不会将过长的输入传递给我们的模型。🤗 Transformers 中的 tokenizer 提供了一个绝妙的 text_target 参数，允许你将目标文本与输入并行 tokenize。以下是如何为 mT5 处理输入和目标文本的示例：

In [ ]:
max_input_length = 512
max_target_length = 30


def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["review_body"],
        max_length=max_input_length,
        truncation=True,
    )
    labels = tokenizer(
        examples["review_title"], max_length=max_target_length, truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

让我们逐步解析这段代码，理解发生了什么。我们首先定义了 max_input_length 和 max_target_length 的值，这些值设定了我们的评论和标题的最大长度。由于评论主体通常比标题大得多，我们相应地调整了这些值。

通过 preprocess_function() 函数，我们可以使用我们在这门课程中广泛使用的方便的 Dataset.map() 函数，轻松地对整个语料库 tokenize 。

In [ ]:
tokenized_datasets = books_dataset.map(preprocess_function, batched=True)

既然语料库已经预处理完毕，我们来看看一些常用的摘要指标。正如我们在下面即将看到的，在衡量机器生成的文本的质量方面没有灵丹妙药。

💡 你可能已经注意到我们在上面的 Dataset.map() 函数中使用了 batched=True 。这将以 1000（默认值）的 batch size 对示例继续编码，并让你可以利用 🤗 Transformers 中快速 tokenizer 的多线程功能。在可能的情况下，尝试使用 batched=True 来加速你的预处理！

文本摘要的评估指标

与我们在本课程中涵盖的大多数其他任务相比，衡量文本生成任务（如摘要或翻译）的好坏并不那么简单。例如，对于“我喜欢阅读饥饿游戏”这样的评论，可能有多个有效摘要，例如“我喜欢饥饿游戏”或“饥饿游戏是一本好书”。显然，在生成的摘要和标签之间进行某种精确匹配并不是一个好的解决方案——即使是人类在这样的评估指标下也会表现不佳，因为每个人都有自己的写作风格。

总而言之，最常用的指标之一是ROUGE 分数（Recall-Oriented Understudy for Gisting Evaluation 的缩写）。该指标背后的基本思想是将生成的摘要与一组通常由人类创建的参考摘要进行比较。更具体地说，假设我们要比较以下两个摘要：

In [ ]:
generated_summary = "I absolutely loved reading the Hunger Games"
reference_summary = "I loved reading the Hunger Games"

比较它们的一种方法是计算重叠单词的数量，在这个例子中为 6。然而，这种方法有些粗糙，因此 ROUGE 是基于计算计算重叠部分的 精确度(Precision) 和 召回率(Recall) 分数来计算的。

🙋 如果这是你第一次听说精确度（Precision）和召回率（Recall），请不要担心——我们将一起通过一些清晰的示例来理解它们。这些指标通常在分类任务中遇到，所以如果你想了解在分类任务中精确度（Precision）和召回率（Recall）是如何定义的，我们建议你查看 scikit-learn 的 指南 。

对于 ROUGE，召回率衡量的是参考摘要中被生成摘要捕获的内容量。如果我们只是比较单词，召回率可以按照以下公式计算：
召回率
=
重叠词的数量/
参考摘要中的总词数

​
 

对于上面的那个例子，这个公式给出了 6/6 = 1 的完美召回率；即，参考摘要中的所有单词模型都生成出来了。这听起来可能很棒，但想象一下，如果我们生成的摘要是“我真的很喜欢整晚阅读饥饿游戏”。这也会有完美的 recall，但可以说这是一个更糟糕的总结，因为它很冗长。为了适应于这些场景，我们还计算了精确度，它在 ROUGE 上下文中衡量了生成的摘要中有多少是相关的：
精确度
=
重叠词的数量/
生成摘要中的总词数

​
 

详细摘要使用这种计算方法会得到 6/10 = 0.6 的精确度，这比较短的摘要获得的 6/7 = 0.86 的精确度要差得多。在实践中，通常会先计算计算精度和召回率，然后得到 F1 分数（精确度和召回率的调和平均数）。我们可以很容易地在🤗 Datasets 中通过安装 rouge_score 包来实现这些计算：

In [ ]:
!pip install rouge_score

然后按如下方式加载 ROUGE 指标：

In [ ]:
import evaluate

rouge_score = evaluate.load("rouge")

接着我们可以使用 rouge_score.compute() 函数来一次性计算所有的指标：

In [ ]:
scores = rouge_score.compute(
    predictions=[generated_summary], references=[reference_summary]
)
scores

哇，这个输出中包含了很多信息——它们都代表什么意思呢？首先，🤗 Datasets 计算了精度、召回率和 F1 分数的置信区间；也些就是你在这里看到的 low 、 mid 和 high 属性。此外，🤗 Datasets 还计算了基于在比较生成摘要和参考摘要时的采用不同文本粒度的各种 ROUGE 得分。 rouge1 测量的是生成摘要和参考摘要中单个单词的重叠程度。 为了验证这一点，让我们提取出我们得分的 mid 值：

In [ ]:
scores["rouge1"].mid

太好了，精确度和召回率的数字都对上了！那么其他的 ROUGE 得分表示什么含义呢？ rouge2 度量了二元词组（考虑单词对的重叠）之间的重叠，而 rougeL 和 rougeLsum 通过寻找生成的摘要和参考摘要中最长的公共子串来度量单词的最长匹配序列。 rougeLsum 中的“sum”指的是该指标是在整个摘要上计算的，而 rougeL 是指在各个句子上计算的平均值。

✏️ 试试看！ 自己手动创建一个生成摘要和参考摘要，看看使用 evaluate 得出的 ROUGE 分数是否与基于精确度和召回率公式的手动计算一致。附加的挑战：将文本切分为长度为2的词组，并手动计算精度和召回率与 rouge2 指标的精确度和召回率进行对比。

我们将使用这些 ROUGE 分数来跟踪我们模型的性能，但在此之前，让我们做每个优秀的 NLP 从业者都应该做的事情：创建一个强大而简单的 baseline！

创建强大的 baseline

对于文本摘要，一个常见的参考 baseline 是简单地取文章的前三句话作为摘要，通常称为 lead-3 baseline。我们可以使用句号（英文使用．）来跟踪句子边界，但这在“U.S.” or “U.N.”之类的首字母缩略词上会计算错误。所以我们将使用 nltk 库，它包含一个更好的算法来处理这些情况。你可以使用以下方式安装该包：

In [ ]:
!pip install nltk

然后下载标点规则：

In [ ]:
import nltk

nltk.download("punkt")

接下来，我们从 nltk 导入句子的 tokenizer 并创建一个简单的函数用来提取评论中的前三个句子。文本摘要的默认情况下使用换行符分隔每个摘要，因此我们也按照这样的规则处理，并在训练集的示例上对其进行测试：

In [ ]:
from nltk.tokenize import sent_tokenize


def three_sentence_summary(text):
    return "\n".join(sent_tokenize(text)[:3])


print(three_sentence_summary(books_dataset["train"][1]["review_body"]))

这似乎有效，接下来让我们现在实现一个函数，从数据集中提取这些“摘要”并计算 baseline 的 ROUGE 分数：

In [ ]:
def evaluate_baseline(dataset, metric):
    summaries = [three_sentence_summary(text) for text in dataset["review_body"]]
    return metric.compute(predictions=summaries, references=dataset["review_title"])

然后我们可以使用这个函数来计算验证集上的 ROUGE 分数，并使用 Pandas 对输出的结果进行一些美化：

In [ ]:
import pandas as pd

score = evaluate_baseline(books_dataset["validation"], rouge_score)
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_dict = dict((rn, round(score[rn].mid.fmeasure * 100, 2)) for rn in rouge_names)
rouge_dict

我们可以看到 rouge2 的分数明显低于其他的rouge；这可能反映了这样一个事实，即评论标题通常很简洁，因此 lead-3 baseline 过于冗长导致得分不高。现在我们有了一个很好的参考基准，让我们将注意力转向微调 mT5！

使用 Trainer API 微调 mT5